In [23]:
import importlib
import plant
importlib.reload (plant)
import numpy as np 
import pandas as pd 

from plant import DroneConfig, DronePlant, RK4_step

In [36]:
config = DroneConfig(
    mass = 2.0, 
    inertia = np.diag([0.01, 0.01, 0.02]), 
    length = 0.2, 
    kd = 0.0, 
    kt = 0.0, 
    kb = 0.0, 
)

y0 = np.zeros (12)
dt = 0.05
t0 = 0.0
t_end = 0.5
g = 9.81

thrust = 1.2 * config.mass * g
torques = np.array ([0.01, 0.0, 0.0])

''' 
* Thrust > gravity force => we expetc the drone to fly up 
* x-component of torques is 0.01 => we should see wx changes first, then phi changes
'''

print (f'Inertia: \n{config.inertia}'); print() 
print (f'length: {config.length}, ', f'mass: {config.mass}'); print()
print (f'RK4 coeffs: kd={config.kd}, kt={config.kt}, kb={config.kb}'); print() 
print (f'init state vec: {y0}'); print()
print (f'start: {t0}, ', f'end: {t_end}, ', f'step: {dt},' , f'time jumps: {(t_end - t0) / dt}'); print()
print (f'thrust: {thrust}'); print() 
print (f'torques: {torques}'); print()

Inertia: 
[[0.01 0.   0.  ]
 [0.   0.01 0.  ]
 [0.   0.   0.02]]

length: 0.2,  mass: 2.0

RK4 coeffs: kd=0.0, kt=0.0, kb=0.0

init state vec: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

start: 0.0,  end: 0.5,  step: 0.05, time jumps: 10.0

thrust: 23.544

torques: [0.01 0.   0.  ]



In [34]:
def f(t, y): 
    plant = DronePlant (config, y) 
    return plant.state_derivatives(thrust, torques)

times = [t0]
state_vec = [y0.copy()]

t = t0 
y = y0.copy() 

while t < t_end:
    y = RK4_step(f, t, y, dt)
    t = round (t + dt, 10)
    times.append (t)
    state_vec.append (y.copy())

state_comp = ['x', 'y', 'z', 'vx', 'vy', 'vz', 'phi', 'theta', 'psi', 'wx', 'wy', 'wz']
df = pd.DataFrame(state_vec, columns = state_comp)
df.insert (0, 'time', times)
df

,time,x,y,z,vx,vy,vz,phi,theta,psi,wx,wy,wz
0,0.00,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.00000,0.0,0.0,0.00,0.0,0.0
1,0.05,0.0,0.000003,-0.002452,0.0,0.000245,-0.098100,0.00125,0.0,0.0,0.05,0.0,0.0
2,0.10,0.0,0.000049,-0.009810,0.0,0.001962,-0.196197,0.00500,0.0,0.0,0.10,0.0,0.0
3,0.15,0.0,0.000248,-0.022072,0.0,0.006622,-0.294278,0.01125,0.0,0.0,0.15,0.0,0.0
4,0.20,0.0,0.000785,-0.039237,0.0,0.015696,-0.392306,0.02000,0.0,0.0,0.20,0.0,0.0
5,0.25,0.0,0.001916,-0.061301,0.0,0.030654,-0.490212,0.03125,0.0,0.0,0.25,0.0,0.0
6,0.30,0.0,0.003973,-0.088254,0.0,0.052966,-0.587885,0.04500,0.0,0.0,0.30,0.0,0.0
7,0.35,0.0,0.007360,-0.120082,0.0,0.084098,-0.685154,0.06125,0.0,0.0,0.35,0.0,0.0
8,0.40,0.0,0.012554,-0.156759,0.0,0.125511,-0.781787,0.08000,0.0,0.0,0.40,0.0,0.0
9,0.45,0.0,0.020106,-0.198245,0.0,0.178656,-0.877472,0.10125,0.0,0.0,0.45,0.0,0.0


Overview:
* wx changes first, while phi changes very trivially
* z gets more negative, meaning drone is flying up

Conclusion:
* The drone behaves as expected, everythign looks kinda correct